<a href="https://colab.research.google.com/github/JoviWZhu/20206RAG/blob/RAG-Hybird/FineWeb_Edu_Hybrid_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets sentence-transformers faiss-cpu rank_bm25 huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 42.2 MB/s eta 0:00:00


In [3]:
import os
import random
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from rank_bm25 import BM25Okapi
from huggingface_hub import InferenceClient

# ==========================================
# 1. INITIALIZATION & SETUP
# ==========================================
# Replace with your free Hugging Face Read Token

from google.colab import userdata
HF_TOKEN_DEV = userdata.get('HF_TOKEN_DEV')

client = InferenceClient(token=HF_TOKEN_DEV)


print("⚡ Step 1: Loading an educational pool from FineWeb-Edu...")
# We load a slightly larger sample (500 documents) so the quiz has diverse topics
dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
#dataset_head = dataset.take(500)

# Pick a random starting point somewhere in the massive dataset stream
random_skip_offset = random.randint(0, 10000)

# Stream 100 rows starting from that random location
dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
dataset_head = dataset.skip(random_skip_offset).take(200)

documents = [doc["text"] for doc in dataset_head]
print(f"✅ Loaded {len(documents)} core documents into your local repository.")


⚡ Step 1: Loading an educational pool from FineWeb-Edu...


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

✅ Loaded 200 core documents into your local repository.


In [4]:
# ==========================================
# 2. BUILD THE DUAL-RETRIEVAL BACKEND
# ==========================================
print("\n⚡ Step 2a: Initializing Semantic Search Channel (Dense Vector)...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
document_embeddings = embedding_model.encode(documents, convert_to_numpy=True)

# Normalize for Cosine Similarity inside an Inner Product space
faiss.normalize_L2(document_embeddings)
dimension = document_embeddings.shape[1]
semantic_index = faiss.IndexFlatIP(dimension)
semantic_index.add(document_embeddings)

print("⚡ Step 2b: Initializing Keyword Search Channel (Sparse BM25)...")
# Tokenize documents into lowercased word arrays for exact token identification
tokenized_corpus = [doc.lower().split(" ") for doc in documents]
bm25_index = BM25Okapi(tokenized_corpus)

print("⚡ Step 2c: Loading BGE Reranker Model onto local CPU memory...")
# This reads query-document sequences and scores contextual overlap via cross-attention matrix
reranker_model = CrossEncoder("BAAI/bge-reranker-base")
print("✅ Hybrid indexing and reranking layers are online.")



⚡ Step 2a: Initializing Semantic Search Channel (Dense Vector)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

✅ Hybrid indexing and reranking layers are online.


In [9]:
print(len(tokenized_corpus[0]))

377


In [16]:
# ==========================================
# 3. CORE HYBRID ROUTING & COMPILATION LOGIC
# ==========================================
def run_hybrid_rag_search(user_query, top_n_candidates=10, final_top_k=2):
    print(f"\n🔍 Processing Hybrid Search Engine Query: '{user_query}'")

    # --- PATHWAY A: DENSE SEMANTIC RETRIEVAL ---
    query_embedding = embedding_model.encode([user_query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    _, semantic_indices = semantic_index.search(query_embedding, top_n_candidates)
    semantic_results = [documents[idx] for idx in semantic_indices[0]]

    # --- PATHWAY B: SPARSE KEYWORD RETRIEVAL ---
    tokenized_query = user_query.lower().split(" ")# Change corpus=documents to documents=documents
    keyword_results = bm25_index.get_top_n(tokenized_query, documents=documents, n=top_n_candidates)


    # --- CANDIDATE POOL FUSION ---
    # Merge both pipelines into a set structure to eliminate duplicate row indexes
    candidate_pool = list(set(semantic_results + keyword_results))
    print(f"✅ Retrieved {len(candidate_pool)} total candidates from sparse & dense channels.")

    # --- STEP 3: CROSS-ATTENTION RERANKING ---
    print("⚡ Reranking candidates using Cross-Encoder attention scoring...")
    rerank_pairs = [[user_query, doc] for doc in candidate_pool]
    rerank_scores = reranker_model.predict(rerank_pairs)

    # Sort document entries in descending order based on their true alignment scores
    ranked_indices = np.argsort(rerank_scores)[::-1]
    final_contexts = [candidate_pool[idx] for idx in ranked_indices[:final_top_k]]
    print(f"🎯 Isolated the top {final_top_k} highest-density context frames.")

    # --- STEP 4: GROUNDED STUDY GUIDE GENERATION ---
    context_str = "\n---\n".join(final_contexts)

    system_prompt = (
        "You are an elite academic curriculum designer. Your task is to process the retrieved textbook fragments "
        "and draft an executive Study Guide for a student. Break down key terms, extract core principles, and "
        "synthesize the facts structured beautifully with markdown headers. Ground everything strictly in the text."
    )

    user_prompt = f"Textbook Context Passages:\n{context_str}\n\nTarget Subject/Concept: {user_query}\n\nDraft Study Guide:"

    print("⚡ Step 4: Routing prompt packet to Hugging Face Serverless endpoint...")
    try:
        response = client.chat_completion(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=800,
            temperature=0.2
        )

        print("\n📚 [ENTERPRISE TEXTBOOK STUDY GUIDE]:")

        # BULLETPROOF PARSING LAYER: Handles object variations across all serverless providers
        if hasattr(response, 'choices') and len(response.choices) > 0:
            choice = response.choices[0]
            if hasattr(choice, 'message'):
                print(choice.message.content)
            elif isinstance(choice, dict) and 'message' in choice:
                print(choice['message']['content'])
            else:
                print(choice)
        else:
            # Direct string/dictionary fallback layout
            if isinstance(response, dict) and 'choices' in response:
                print(response['choices'][0]['message']['content'])
            else:
                print(response)

    except Exception as e:
        print(f"❌ Error communicating with Hugging Face API: {e}")

In [17]:
run_hybrid_rag_search("What are the definitions and core characteristics of physical properties or engineering frameworks?", final_top_k=2)


🔍 Processing Hybrid Search Engine Query: 'What are the definitions and core characteristics of physical properties or engineering frameworks?'
✅ Retrieved 19 total candidates from sparse & dense channels.
⚡ Reranking candidates using Cross-Encoder attention scoring...
🎯 Isolated the top 2 highest-density context frames.
⚡ Step 4: Routing prompt packet to Hugging Face Serverless endpoint...

📚 [ENTERPRISE TEXTBOOK STUDY GUIDE]:
**Study Guide: Inertial Frames and Reference Frames**

**I. Introduction**

* Inertial frames and reference frames are fundamental concepts in physics and engineering.
* Understanding these concepts is crucial for analyzing motion, forces, and energy.

**II. Key Terms and Definitions**

* **Inertial Frame:** A reference frame in which a body not subject to forces travels in a straight line with a uniform velocity.
* **Reference Frame:** A standard relative to which motion and rest may be measured.
* **Galilean Relativity:** The principle that mechanical experime